In [1]:
# Install necessary libraries
# We update torch, torchvision, and torchaudio TOGETHER to avoid version mismatches
!pip install -q -U torch torchvision torchaudio transformers bitsandbytes accelerate

# Install LangChain and RAG tools
!pip install -q langchain langchain-community langchain-huggingface langchain-chroma langchain-text-splitters

# Install Utilities and App tools (Moved from later cells)
!pip install -q pypdf sentence-transformers gradio flask flask-cors pyngrok

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.1.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.36.2 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-huggingface 1.2.0 requires huggingface-hub<1.0.0,>=0.33.4, but you have huggingface-hub 1.4.1 which is incompatible.


In [ ]:
from huggingface_hub import login

# --- AUTOMATIC LOGIN ---
# 1. Get your token from: https://huggingface.co/settings/tokens
# 2. Ensure it is a "READ" token.
# 3. Paste it below in place of the placeholder text.

my_hf_token = "PASTE_YOUR_TOKEN_HERE"

# This logs you in immediately using the variable
login(token=my_hf_token)

print("--- logged in successfully ---")

--- logged in successfully ---


In [3]:
import torch
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings

# --- 1. MODEL SELECTION ---
STARTUP_MODEL = "llama3"

model_map = {
    "llama3": "meta-llama/Meta-Llama-3-8B-Instruct",
}

model_id = model_map.get(STARTUP_MODEL)
print(f"--- Loading {STARTUP_MODEL.upper()} ({model_id}) ---")

# --- 2. SETUP EMBEDDINGS ---
embedding_model_name = "sentence-transformers/all-mpnet-base-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)

# --- 3. SETUP MODEL & TOKENIZER ---
print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

terminators = None
if STARTUP_MODEL == "llama3":
    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

print("Loading Model (4-bit)...")

# --- THE FIX: Use BitsAndBytesConfig ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,  # Pass the config here
    device_map="auto",
    trust_remote_code=True
)

# --- 4. CREATE PIPELINE ---
text_generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    eos_token_id=terminators,
    do_sample=True,
    temperature=0.6,
    top_p=0.9,
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

print(f"--- {STARTUP_MODEL.upper()} Loaded Successfully ---")

--- Loading LLAMA3 (meta-llama/Meta-Llama-3-8B-Instruct) ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading Tokenizer...
Loading Model (4-bit)...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'top_p', 'max_new_tokens', 'do_sample', 'eos_token_id', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


--- LLAMA3 Loaded Successfully ---


In [4]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.documents import Document
from huggingface_hub import snapshot_download

# --- 1. DOWNLOAD FILES FROM YOUR HUGGING FACE SPACE ---
print("--- Downloading Files from Hugging Face Space ---")
# This downloads your entire space to a local folder in Colab
repo_id = "kydeleon/RTAverse.AI"  # Your Space ID
local_dir = "/content/rtaverse_files"

snapshot_download(
    repo_id=repo_id,
    repo_type="space",
    local_dir=local_dir,
    token=my_hf_token  # Uses the token you set in Block 2
)
print("Files downloaded successfully.")

# --- Configuration ---
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
# Update path to where we downloaded the files
DATA_PATH = local_dir

def load_documents():
    documents = []

    # 2. Load PDF (Assuming it is inside the 'data' folder of your Space)
    # Check both root and data folder just in case
    pdf_paths = [
        os.path.join(DATA_PATH, "data", "rtaverse_paper.pdf"),
        os.path.join(DATA_PATH, "rtaverse_paper.pdf")
    ]

    pdf_found = False
    for pdf_path in pdf_paths:
        if os.path.exists(pdf_path):
            print(f"Loading PDF: {pdf_path}")
            loader = PyPDFLoader(pdf_path)
            documents.extend(loader.load())
            pdf_found = True
            break

    if not pdf_found:
        print("Warning: rtaverse_paper.pdf not found in the Space.")

    # 3. Load Source Code (Walking through the downloaded repo)
    # We scan the WHOLE downloaded folder for code files
    extensions = (".py", ".html", ".css", ".js", ".java", ".cpp")
    exclude_folders = {".git", "__pycache__", "chroma_db"} # Skip system folders

    print("Loading source code files...")
    for root, dirs, files in os.walk(DATA_PATH):
        # Filter out unwanted folders to speed it up
        dirs[:] = [d for d in dirs if d not in exclude_folders]

        for file in files:
            if file.endswith(extensions):
                file_path = os.path.join(root, file)
                try:
                    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                        content = f.read()
                        # Add metadata so the AI knows which file this came from
                        documents.append(Document(page_content=content, metadata={"source": file}))
                except Exception as e:
                    print(f"Error reading {file}: {e}")

    return documents

# --- Execution ---
print("Loading Documents...")
docs = load_documents()

if not docs:
    print("CRITICAL WARNING: No documents found. creating a dummy document.")
    docs = [Document(page_content="RTAverse is a thesis project.", metadata={"source": "dummy"})]

print(f"Splitting {len(docs)} documents...")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
splits = text_splitter.split_documents(docs)

print("Creating Vector Database...")
# Force a new DB to ensure it has the latest files
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print("--- RAG Chain Updated with Hugging Face Files ---")

--- Downloading Files from Hugging Face Space ---


Fetching 63 files:   0%|          | 0/63 [00:00<?, ?it/s]

Files downloaded successfully.
Loading Documents...
Loading PDF: /content/rtaverse_files/data/rtaverse_paper.pdf
Loading source code files...
Splitting 150 documents...
Creating Vector Database...
--- RAG Chain Updated with Hugging Face Files ---


In [5]:
import os

# Create necessary Flask directories
os.makedirs('templates', exist_ok=True)
os.makedirs('static', exist_ok=True)

# --- 1. WRITE INDEX.HTML ---
# Updated to include Mistral and Gemma options
html_content = """
<!DOCTYPE html>
<html lang="en">
  <head>
    <meta charset="UTF-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0" />
    <title>RTAverse AI</title>
    <link
      href="https://api.fontshare.com/v2/css?f[]=chillax@300,400,500,600,700&display=swap"
      rel="stylesheet"
    />
    <link
      rel="stylesheet"
      href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css"
    />
    <link rel="stylesheet" href="/static/style.css" />
  </head>
  <body class="dark-mode">
    <aside class="sidebar">
      <div class="sidebar-top">
        <div class="sidebar-header">
          <div class="logo-area">
            <i class="fa-solid fa-gem logo-icon"></i>
            <h1>RTAverse AI</h1>
          </div>
        </div>
        <div class="sidebar-content">
          <div class="control-group">
  <label>Foundation Model</label>
  <div style="padding: 12px; background: rgba(255, 255, 255, 0.05); border: 1px solid rgba(255, 255, 255, 0.1); border-radius: 12px; font-size: 0.95rem;">
    <i class="fa-solid fa-bolt" style="color: var(--accent-color); margin-right: 8px;"></i>
    Llama 3 (Cloud)
  </div>

  <input type="hidden" id="model-select" value="llama3" />
</div>
            <label>Chunk Size: <span id="chunk-val">1000</span></label>
            <input type="range" id="chunk-slider" min="200" max="2000" step="100" value="1000" />
            <button id="apply-chunk" class="small-btn">Apply Re-index</button>
          </div>
          <div class="control-group toggle-area">
            <label>Appearance</label>
            <button id="theme-toggle" class="theme-btn">
              <i class="fa-solid fa-moon"></i>
              <span>Switch Theme</span>
            </button>
          </div>
        </div>
      </div>
      <div class="sidebar-footer">
        <div class="status-item"><span class="dot green"></span> Chain Ready</div>
        <div class="status-item"><span class="dot green"></span> Docs Loaded</div>
        <div class="status-item database-info">DB: Chroma (Colab)</div>
      </div>
    </aside>
    <main class="main-panel">
      <div id="welcome-screen" class="welcome-container">
        <h2>How can I help with your Thesis?</h2>
        <p>I have access to the RTAverse paper and source code. Ask me about architecture, algorithms, or specific implementations.</p>
      </div>
      <div id="chat-container" class="chat-container"></div>
      <div class="input-area">
        <div class="input-wrapper">
          <input type="text" id="user-input" placeholder="Ask about the thesis or code..." autocomplete="off" />
          <button id="send-btn"><i class="fa-solid fa-paper-plane"></i></button>
        </div>
      </div>
    </main>
    <script>
      const chatContainer = document.getElementById("chat-container");
      const userInput = document.getElementById("user-input");
      const sendBtn = document.getElementById("send-btn");
      const welcomeScreen = document.getElementById("welcome-screen");
      const modelSelect = document.getElementById("model-select");
      const themeToggle = document.getElementById("theme-toggle");
      const chunkSlider = document.getElementById("chunk-slider");
      const chunkVal = document.getElementById("chunk-val");
      const applyChunkBtn = document.getElementById("apply-chunk");

      chunkSlider.addEventListener("input", (e) => { chunkVal.textContent = e.target.value; });

      applyChunkBtn.addEventListener("click", async () => {
        const size = chunkSlider.value;
        applyChunkBtn.textContent = "Indexing...";
        applyChunkBtn.disabled = true;
        try {
          const response = await fetch("/update_settings", {
            method: "POST",
            headers: { "Content-Type": "application/json" },
            body: JSON.stringify({ chunk_size: size }),
          });
          const data = await response.json();
          alert(data.message);
        } catch (e) { alert("Error updating settings"); }
        finally { applyChunkBtn.textContent = "Apply Re-index"; applyChunkBtn.disabled = false; }
      });

      themeToggle.addEventListener("click", () => {
        document.body.classList.toggle("light-mode");
        const icon = themeToggle.querySelector("i");
        if (document.body.classList.contains("light-mode")) {
          icon.classList.remove("fa-moon"); icon.classList.add("fa-sun");
        } else {
          icon.classList.remove("fa-sun"); icon.classList.add("fa-moon");
        }
      });

      function appendMessage(text, sender, sources = []) {
        welcomeScreen.style.display = "none";
        const msgDiv = document.createElement("div");
        msgDiv.classList.add("message", sender);
        let contentHtml = `<div class="bubble">${formatText(text)}</div>`;
        if (sender === "bot") {
          if (sources.length > 0) contentHtml += `<div class="sources">Sources: ${sources.join(", ")}</div>`;
          contentHtml += `<button class="summarize-btn" onclick="summarizeText(this)"><i class="fa-solid fa-wand-magic-sparkles"></i> Summarize this</button>`;
        }
        msgDiv.innerHTML = contentHtml;
        chatContainer.appendChild(msgDiv);
        chatContainer.scrollTop = chatContainer.scrollHeight;
      }

      function formatText(text) { return text.replace(/\\n/g, "<br>"); }

      async function summarizeText(btn) {
        const messageDiv = btn.parentElement;
        const textBubble = messageDiv.querySelector(".bubble");
        const originalText = textBubble.innerText;
        btn.innerHTML = `<i class="fa-solid fa-circle-notch fa-spin"></i> Summarizing...`;
        btn.disabled = true;
        try {
          const response = await fetch("/summarize", {
            method: "POST",
            headers: { "Content-Type": "application/json" },
            body: JSON.stringify({ text: originalText, model: modelSelect.value }),
          });
          const data = await response.json();
          const summaryDiv = document.createElement("div");
          summaryDiv.classList.add("summary-box");
          summaryDiv.innerHTML = `<strong>Summary:</strong> ${data.summary}`;
          messageDiv.appendChild(summaryDiv);
          btn.style.display = "none";
        } catch (error) { btn.innerHTML = `<i class="fa-solid fa-triangle-exclamation"></i> Error`; }
      }

      function showLoading() {
        const loader = document.createElement("div");
        loader.id = "loader";
        loader.classList.add("message", "bot");
        loader.innerHTML = `<div class="bubble loading-bubble"><span class=\"dot\"></span><span class=\"dot\"></span><span class=\"dot\"></span></div>`;
        chatContainer.appendChild(loader);
        chatContainer.scrollTop = chatContainer.scrollHeight;
      }

      function removeLoading() { const loader = document.getElementById("loader"); if (loader) loader.remove(); }

      async function sendMessage() {
        const text = userInput.value.trim();
        if (!text) return;
        appendMessage(text, "user");
        userInput.value = "";
        showLoading();
        try {
          const response = await fetch("/ask", {
            method: "POST",
            headers: { "Content-Type": "application/json" },
            body: JSON.stringify({ question: text, model: modelSelect.value }),
          });
          const data = await response.json();
          removeLoading();
          if (data.error) appendMessage("Error: " + data.error, "bot");
          else appendMessage(data.answer, "bot", data.sources);
        } catch (error) { removeLoading(); appendMessage("Network Error: Could not reach backend.", "bot"); }
      }

      sendBtn.addEventListener("click", sendMessage);
      userInput.addEventListener("keypress", (e) => { if (e.key === "Enter") sendMessage(); });
    </script>
  </body>
</html>
"""

# --- 2. WRITE STYLE.CSS ---
# (Keeping your original CSS style logic)
# --- 2. WRITE STYLE.CSS ---
css_content = """
:root {
  --sidebar-bg: #0b1120;
  --main-bg: #1e293b;
  --text-main: #f1f5f9;
  --text-muted: #94a3b8;
  --text-sidebar-head: #ffffff;
  --accent-color: #3b82f6;
  --accent-hover: #2563eb;
  --accent-glow: rgba(59, 130, 246, 0.3);
  --bubble-user: #3b82f6;
  --bubble-bot: #334155;
  --bubble-text-bot: #f1f5f9;
  --input-bg: rgba(15, 23, 42, 0.8);
  --input-border: rgba(255, 255, 255, 0.08);
  --glass-bg: linear-gradient(to top, rgba(30, 41, 59, 1) 20%, rgba(30, 41, 59, 0));
  --border-radius: 24px;
  --pill-radius: 99px;
  --font-primary: "Chillax", sans-serif;
}
body.light-mode {
  --sidebar-bg: #f8fafc;
  --main-bg: #ffffff;
  --text-main: #0f172a;
  --text-muted: #64748b;
  --text-sidebar-head: #0f172a;
  --accent-color: #2563eb;
  --accent-hover: #1d4ed8;
  --accent-glow: rgba(37, 99, 235, 0.15);
  --bubble-user: #2563eb;
  --bubble-bot: #f1f5f9;
  --bubble-text-bot: #334155;
  --input-bg: rgba(255, 255, 255, 0.9);
  --input-border: #e2e8f0;
  --glass-bg: linear-gradient(to top, rgba(255, 255, 255, 1) 20%, rgba(255, 255, 255, 0));
}
* { margin: 0; padding: 0; box-sizing: border-box; }
body { font-family: var(--font-primary); background-color: var(--main-bg); color: var(--text-main); display: flex; height: 100vh; overflow: hidden; font-size: 16px; transition: background-color 0.3s ease, color 0.3s ease; }
.sidebar { width: 300px; background-color: var(--sidebar-bg); padding: 2.5rem 2rem; display: flex; flex-direction: column; border-right: 1px solid var(--input-border); transition: background-color 0.3s ease, border-color 0.3s ease; z-index: 20; }
.sidebar-header { margin-bottom: 2.5rem; }
.sidebar-header .logo-area { display: flex; align-items: center; gap: 12px; }
.logo-icon { color: var(--accent-color); font-size: 1.4rem; filter: drop-shadow(0 0 8px var(--accent-glow)); }
.sidebar h1 { font-weight: 600; font-size: 1.4rem; color: var(--text-sidebar-head); letter-spacing: 0.02em; }
.sidebar-content { display: flex; flex-direction: column; gap: 2rem; }
.control-group label { display: block; font-size: 0.75rem; font-weight: 700; text-transform: uppercase; letter-spacing: 0.08em; margin-bottom: 0.8rem; color: var(--text-muted); }

/* --- NEW SPACING RULE --- */
.toggle-area {
  margin-top: 1.5rem;          /* Adds extra space above the section */
  padding-top: 2rem;           /* Pushes the content down internally */
  border-top: 1px solid rgba(255, 255, 255, 0.1); /* Adds a subtle separator line */
}

.custom-select-wrapper { position: relative; width: 100%; }
select { width: 100%; padding: 14px 45px 14px 20px; background-color: rgba(255, 255, 255, 0.03); border: 1px solid var(--input-border); color: var(--text-sidebar-head); border-radius: var(--border-radius); appearance: none; -webkit-appearance: none; outline: none; cursor: pointer; font-family: var(--font-primary); font-size: 0.95rem; font-weight: 500; transition: all 0.2s ease; }
select:focus { border-color: var(--accent-color); box-shadow: 0 0 0 3px var(--accent-glow); }
select option { background-color: var(--sidebar-bg); color: var(--text-main); padding: 10px; }
.select-icon { position: absolute; right: 20px; top: 50%; transform: translateY(-50%); pointer-events: none; font-size: 0.8rem; color: var(--text-muted); }
input[type="range"] { width: 100%; margin-bottom: 12px; accent-color: var(--accent-color); cursor: pointer; height: 6px; border-radius: 4px; background: rgba(255, 255, 255, 0.1); }
.small-btn { width: 100%; padding: 12px; background: transparent; border: 1px solid var(--accent-color); color: var(--accent-color); border-radius: var(--pill-radius); cursor: pointer; font-family: var(--font-primary); font-weight: 600; font-size: 0.85rem; transition: all 0.2s ease; }
.small-btn:hover { background: var(--accent-color); color: #fff; box-shadow: 0 4px 15px var(--accent-glow); transform: translateY(-1px); }
.small-btn:disabled { opacity: 0.5; cursor: not-allowed; }
.theme-btn { width: 100%; padding: 14px; display: flex; align-items: center; justify-content: center; gap: 12px; background: rgba(255, 255, 255, 0.03); border: 1px solid var(--input-border); border-radius: var(--pill-radius); color: var(--text-sidebar-head); cursor: pointer; font-family: var(--font-primary); font-size: 0.9rem; font-weight: 500; transition: all 0.2s; }
.theme-btn:hover { background: rgba(255, 255, 255, 0.06); border-color: var(--text-muted); }
.sidebar-footer { margin-top: auto; padding-top: 2rem; font-size: 0.75rem; color: var(--text-muted); border-top: 1px solid rgba(255, 255, 255, 0.05); }
.status-item { display: flex; align-items: center; gap: 8px; margin-bottom: 8px; font-weight: 500; }
.dot { width: 6px; height: 6px; border-radius: 50%; background-color: var(--text-muted); }
.dot.green { background-color: #10b981; box-shadow: 0 0 6px rgba(16, 185, 129, 0.4); }
.main-panel { flex: 1; display: flex; flex-direction: column; position: relative; max-width: 1600px; margin: 0 auto; width: 100%; height: 100%; }
.welcome-container { position: absolute; top: 45%; left: 50%; transform: translate(-50%, -50%); text-align: center; width: 100%; padding: 0 2rem; z-index: 0; }
.welcome-container h2 { font-size: 3rem; font-weight: 600; margin-bottom: 1.2rem; background: linear-gradient(135deg, var(--text-main) 30%, var(--text-muted) 100%); -webkit-background-clip: text; -webkit-text-fill-color: transparent; }
.welcome-container p { font-size: 1.15rem; color: var(--text-muted); max-width: 600px; margin: 0 auto; line-height: 1.6; }
.chat-container { flex: 1; padding: 2rem 15% 10rem 15%; overflow-y: auto; display: flex; flex-direction: column; gap: 1.5rem; scroll-behavior: smooth; z-index: 1; }
.message { display: flex; flex-direction: column; max-width: 80%; animation: fadeIn 0.4s cubic-bezier(0.25, 1, 0.5, 1); }
.message.user { align-self: flex-end; align-items: flex-end; }
.message.bot { align-self: flex-start; align-items: flex-start; }
.bubble { padding: 1.2rem 1.8rem; border-radius: var(--border-radius); line-height: 1.6; font-size: 1rem; position: relative; box-shadow: 0 2px 5px rgba(0, 0, 0, 0.03); }
.message.user .bubble { background-color: var(--bubble-user); color: #ffffff; border-bottom-right-radius: 4px; }
.message.bot .bubble { background-color: var(--bubble-bot); color: var(--bubble-text-bot); border-bottom-left-radius: 4px; border: 1px solid var(--input-border); }
.sources { font-size: 0.75rem; color: var(--text-muted); margin-top: 8px; margin-left: 12px; display: flex; flex-wrap: wrap; gap: 8px; }
.summarize-btn { background: none; border: none; color: var(--accent-color); font-family: var(--font-primary); font-weight: 600; font-size: 0.8rem; margin-top: 10px; margin-left: 5px; cursor: pointer; opacity: 0.8; transition: all 0.2s; display: inline-flex; align-items: center; gap: 6px; padding: 4px 8px; border-radius: 8px; }
.summarize-btn:hover { opacity: 1; color: var(--accent-hover); background: rgba(59, 130, 246, 0.05); }
.summary-box { margin-top: 15px; background: rgba(125, 125, 125, 0.05); padding: 16px; border-radius: var(--border-radius); font-size: 0.95rem; line-height: 1.6; border-left: 4px solid var(--accent-color); color: var(--text-main); }
.input-area { position: absolute; bottom: 0; left: 0; width: 100%; padding: 2rem; padding-bottom: 3rem; background: var(--glass-bg); backdrop-filter: blur(8px); display: flex; justify-content: center; z-index: 10; pointer-events: none; }
.input-wrapper { position: relative; width: 100%; max-width: 800px; pointer-events: auto; filter: drop-shadow(0 10px 20px rgba(0, 0, 0, 0.15)); }
.input-wrapper input { width: 100%; padding: 1.4rem 4.5rem 1.4rem 2.2rem; background-color: var(--input-bg); border: 1px solid var(--input-border); border-radius: var(--pill-radius); color: var(--text-main); font-family: var(--font-primary); font-size: 1.05rem; backdrop-filter: blur(12px); transition: all 0.3s ease; }
.input-wrapper input:focus { outline: none; border-color: var(--accent-color); box-shadow: 0 0 0 4px var(--accent-glow); background-color: var(--sidebar-bg); }
#send-btn { position: absolute; right: 12px; top: 50%; transform: translateY(-50%); width: 48px; height: 48px; border-radius: 50%; background-color: var(--accent-color); border: none; color: white; font-size: 1.1rem; cursor: pointer; transition: all 0.2s cubic-bezier(0.34, 1.56, 0.64, 1); display: flex; align-items: center; justify-content: center; }
#send-btn:hover { background-color: var(--accent-hover); transform: translateY(-50%) scale(1.08); }
@keyframes fadeIn { from { opacity: 0; transform: translateY(15px); } to { opacity: 1; transform: translateY(0); } }
.loading-bubble .dot { display: inline-block; width: 7px; height: 7px; background-color: var(--text-muted); border-radius: 50%; margin: 0 3px; animation: bounce 1.4s infinite ease-in-out both; }
.loading-bubble .dot:nth-child(1) { animation-delay: -0.32s; }
.loading-bubble .dot:nth-child(2) { animation-delay: -0.16s; }
@keyframes bounce { 0%, 80%, 100% { transform: scale(0); } 40% { transform: scale(1); } }
::-webkit-scrollbar { width: 8px; }
::-webkit-scrollbar-track { background: transparent; }
::-webkit-scrollbar-thumb { background: rgba(150, 150, 150, 0.2); border-radius: 4px; }
::-webkit-scrollbar-thumb:hover { background: rgba(150, 150, 150, 0.4); }
"""

with open('static/style.css', 'w') as f:
    f.write(css_content)

print("Updated style.css with new spacing.")

Updated style.css with new spacing.


In [ ]:
from flask import Flask, render_template, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import threading
import os
import re
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

# ---------------- CONFIGURATION ----------------
NGROK_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"
app = Flask(__name__)
CORS(app)

# Global variables
global rag_chain, vectorstore, docs
rag_chain = None
vectorstore = None
current_config = {"chunk_size": 1000, "chunk_overlap": 200}

# --- DEFINE PATH AND LOAD PDF (Safety Check) ---
pdf_path = "/content/rtaverse_files/data/rtaverse_paper.pdf"

# If 'docs' isn't already loaded from Cell 6, load it now
if 'docs' not in globals() or not docs:
    print(f"Docs not found in memory. Checking file: {pdf_path}")
    if os.path.exists(pdf_path):
        loader = PyPDFLoader(pdf_path)
        docs = loader.load()
        print(f"Successfully loaded {len(docs)} pages.")
    else:
        print("WARNING: rtaverse_paper.pdf not found. Please run Cell 6 to download files.")
        docs = []

def initialize_rag_chain(chunk_size=1000):
    global rag_chain, vectorstore, current_config, llm, embeddings, docs

    print(f"--- Initializing Chain (Chunk: {chunk_size}) ---")
    current_config["chunk_size"] = chunk_size

    if not docs:
        print("Warning: No documents loaded. Chain will be empty.")
        return

    # Split docs
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=200)
    splits = text_splitter.split_documents(docs)

    # Re-create Vector Store
    vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

    # --- PROMPT TEMPLATE ---
    template = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are the intelligent AI assistant for **RTAverse**, a **Machine Learning-based Road Traffic Accident Forecasting System** for Angeles City.

Your goal is to help users understand traffic patterns, accident forecasting, and data visualization.

CRITICAL INSTRUCTIONS:
1. **Core Identity:** RTAverse is about *traffic safety and prediction*, NOT just a software table or a secure server.
2. **Ignore Infrastructure:** If the 'CONTEXT' below discusses "DDoS protection," "Render," "Ngrok," "Private Networking," or "Git," **IGNORE IT** unless the user specifically asks "How is this hosted?" or "Is it secure?".
3. **Prioritize Purpose:** If asked for "Objectives," focus on the *study's* objectives (reducing accidents, forecasting hotspots) rather than the *security* objectives.
4. **Be Direct:** Answer the question directly. Do not say "Based on the context."

<|eot_id|><|start_header_id|>user<|end_header_id|>

CONTEXT (Use this info, but ignore hosting/server details if irrelevant):
{context}

QUESTION:
{question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

    prompt = PromptTemplate.from_template(template)

    rag_chain = (
        {"context": retriever, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    print("--- Chain Ready ---")

def clean_llama_output(text):
    if "assistant<|end_header_id|>" in text:
        text = text.split("assistant<|end_header_id|>")[-1]
    elif "<|start_header_id|>assistant" in text:
        text = text.split("<|start_header_id|>assistant")[-1]
    text = text.replace("<|begin_of_text|>", "").replace("<|eot_id|>", "").replace("<|end_header_id|>", "")
    return text.strip()

# Initialize once on startup
initialize_rag_chain()

# ---------------- ROUTES ----------------
@app.route('/')
def index():
    return render_template('index.html')

@app.route('/ask', methods=['POST'])
def ask():
    data = request.get_json()
    question = data.get('question')

    if not rag_chain: return jsonify({"error": "Initializing..."}), 503

    try:
        response = rag_chain.invoke(question)
        retriever = vectorstore.as_retriever()
        source_docs = retriever.invoke(question)
        sources = list(set([os.path.basename(doc.metadata.get("source", "Unknown")) for doc in source_docs]))
        final_answer = clean_llama_output(response)
        return jsonify({"answer": final_answer, "sources": sources})
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/summarize', methods=['POST'])
def summarize():
    data = request.get_json()
    text_to_summarize = data.get('text')

    try:
        prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a research assistant for the RTAverse Thesis Project. Summarize the text below in 2-3 concise sentences.
STRICT GUIDELINES: Start IMMEDIATELY. Focus on traffic/forecasting. IGNORE infrastructure/hosting unless it is the only text.
<|eot_id|><|start_header_id|>user<|end_header_id|>
TEXT: {text_to_summarize}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

        raw_summary = llm.invoke(prompt)
        final_summary = clean_llama_output(raw_summary)
        return jsonify({"summary": final_summary.strip()})
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/update_settings', methods=['POST'])
def update_settings():
    data = request.get_json()
    new_size = int(data.get('chunk_size', 1000))
    initialize_rag_chain(chunk_size=new_size)
    return jsonify({"message": f"Re-indexed with chunk size {new_size}"})

# ---------------- RUNNER ----------------
if __name__ == '__main__':
    # Kill any existing ngrok tunnels to avoid "too many tunnels" error
    ngrok.kill()
    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(5000)
    print(f"\n\n CLICK THIS LINK TO OPEN YOUR APP: {public_url.public_url} \n\n")
    app.run(port=5000)